In [1]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
import deepchem as dc
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, OneCycleLR

from data_loader import get_dataset
from model import GNN
from copy import deepcopy
import  torch_geometric.transforms as T 

Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading some Jax models, missing a dependency. No module named 'jax'


In [2]:
import random
import numpy as np
import torch
import torch_geometric
def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    np.random.seed(seed)
    random.seed(seed)
    torch_geometric.seed_everything(seed)

In [3]:
def train(args, filename=None):
    with mlflow.start_run() as run:
        run_id = run.info.run_id
        mlflow.log_params(vars(args))
        if filename:
            mlflow.log_artifact( filename, artifact_path='code')     
        if args.gpu >= 0:
            device = torch.device('cuda:%d' % args.gpu)
        else:
            device = torch.device('cpu')
            
        maes, mapes, mses = [], [], []
        best_vals = []
        for trial in range(args.num_trial):
            setup_seed(trial)
            
            if 'VN' in args.model:
                transform = T.VirtualNode()
            else:
                transform = None
                
            dataloader,  dataloader_test, dataloader_val, transformer, meta = get_dataset(args, transform)
            num_classes = meta['num_classes']
            n_train = len(dataloader.dataset)
            n_val = len(dataloader_val.dataset)
            n_test = len(dataloader_test.dataset) 
            # Model initialization
            
            model = GNN(num_classes, args.hidden_dim,args.num_layer, conv=args.model, pool=args.pool, norm = 'BatchNorm' 
                        if args.bn else args.norm, l2norm=args.l2norm, dropout=args.dropout, criterion = args.criterion, jk=args.jk, normalize=args.normalize, 
                        first_residual=args.first_residual, residual=args.residual, final_batch_norm=args.final_batch_norm).to(device)
            
            optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
            if args.scheduler.startswith('step'):
                step_size, gamma = args.scheduler.split('-')[1:]
                scheduler = StepLR(optimizer, step_size=int(step_size), gamma=float(gamma))
            elif args.scheduler == 'cosine':
                scheduler = CosineAnnealingLR(optimizer, T_max=args.num_epoch) 
            elif args.scheduler.startswith('onecycle'):
                pct_start = float(args.scheduler.split('-')[1]) if '-' in args.scheduler else 0.1
                scheduler = OneCycleLR(optimizer, max_lr=args.lr, steps_per_epoch=len(dataloader), epochs=args.num_epoch, pct_start=pct_start, final_div_factor=args.final_div_factor)
            else:
                scheduler = None
            # Training&Validation
            best_val = float("Inf")
            best_epoch = 0
            for epoch in range(1, args.num_epoch + 1):
                model.train()
                loss_all = 0
                for data in dataloader:
                    data = data.to(device)
                    optimizer.zero_grad()
                    loss = model.calc_loss(data)
                    loss_all += loss.item() * data.num_graphs
                    loss.backward()
                    optimizer.step()
                if scheduler is not None:
                    scheduler.step()
#                 print('[TRAIN] Epoch:{:03d} | Loss:{:.4f}'.format(epoch, loss_all / n_train))
                mlflow.log_metric('Loss', loss_all / n_train, step=epoch)
                # Validation
                model.eval()
                loss_all_val = 0.0                
                with torch.no_grad():            
                    for data in dataloader_val:
                        data = data.to(device)
                        loss = model.calc_loss(data)
                        loss_all_val += loss.item() * data.num_graphs
                mlflow.log_metric('Loss_val', loss_all_val / n_val, step=epoch)
                if loss_all_val < best_val:
                    best_val = loss_all_val
                    best_model = deepcopy(model.state_dict())
                    best_epoch = epoch  

                if epoch % args.eval_freq == 0:
                    model.eval()
                    y_true = []
                    y_preds = []                   
                    with torch.no_grad():
                        for data in dataloader_test:
                            data = data.to(device)
                            y_true = y_true + data.y.cpu().reshape(-1, num_classes).tolist()
                            y_preds = y_preds + model.predict_score(data).cpu().reshape(-1, num_classes).tolist()
                    if args.normalize:
                        y_true= transformer.inverse_transform(y_true)
                        y_preds = transformer.inverse_transform(y_preds) 
                    mae = mean_absolute_error(y_true, y_preds)
                    mape = mean_absolute_percentage_error(y_true, y_preds)
                    mse = mean_squared_error(y_true, y_preds)
                    mlflow.log_metrics({'MAE_epoch': mae, 'MAPE_epoch': mape, 'MSE_epoch': mse}, step=epoch)                                    
            # Test on best validation
            model.load_state_dict(best_model) 
            model.eval()
            y_true = []
            y_preds = []
            with torch.no_grad():     
                for data in dataloader_test:
                    data = data.to(device)
                    y_true = y_true + data.y.cpu().reshape(-1, num_classes).tolist()
                    y_preds = y_preds + model.predict_score(data).cpu().reshape(-1, num_classes).tolist()
            assert len(y_true) == n_test and len(y_preds) == n_test
            if args.normalize:
                y_true= transformer.inverse_transform(y_true)
                y_preds = transformer.inverse_transform(y_preds)                     
            mae = mean_absolute_error(y_true, y_preds)
            mape = mean_absolute_percentage_error(y_true, y_preds)
            mse = mean_squared_error(y_true, y_preds)     
            mlflow.log_metrics({'MAE_trial': mae, 'MAPE_trial': mape, 'MSE_trial': mse}, step=trial)  
            maes.append(mae)
            mapes.append(mape)
            mses.append(mse)
            mlflow.log_metric(f'Best_epoch', best_epoch, step=trial)
            mlflow.log_metric(f'Best_Val', best_val, step=trial)
            best_vals.append(best_val)

        # Log average results
        avg_val = np.mean(best_vals)
        std_val = np.std(best_vals)
        mlflow.log_metric(f'Best_Val_mean', avg_val)
        mlflow.log_metric(f'Best_Val_std', std_val)        
        
        avg_val = np.mean(maes)
        std_val = np.std(maes)
        mlflow.log_metric(f'MAE_mean', avg_val)
        mlflow.log_metric(f'MAE_std', std_val)      
        
        avg_val = np.mean(mapes)
        std_val = np.std(mapes)
        mlflow.log_metric(f'MAPE_mean', avg_val)
        mlflow.log_metric(f'MAPE_std', std_val)
        
        avg_val = np.mean(mses)
        std_val = np.std(mses)
        mlflow.log_metric(f'MSE_mean', avg_val)
        mlflow.log_metric(f'MSE_std', std_val)
          
            

In [ ]:

%%capture
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

import mlflow
import argparse
exp_name = 'GNN_Baselines_build'
mlflow.set_tracking_uri('file:///data/leylazhang_proj/PCE/mlruns')
try:
    mlflow.create_experiment(exp_name)
except:
    print("Experiment has been created")
mlflow.set_experiment(exp_name)


parser = argparse.ArgumentParser()
parser.add_argument('-dataset', type=str, default='PolymerFA', choices=['NFAs_51K','NFAs_1.2K','HOPV', 'PolymerFA', 'nNFA', 'pNFA'])
parser.add_argument('-featurizer', type=str, default=None, choices=[None, 'MACCS', 'ECFP6', 'Mordred'])
parser.add_argument('-normalize', type=bool, default=False)
parser.add_argument('-scaler', type=str, default='standard', choices=['minmax', 'standard'])
parser.add_argument('-frac_train', type=float, default=0.6)
parser.add_argument('-target_mode', type=str, default='single')
parser.add_argument('-target_task', type=int, default=0, help='0: PCE, 1: HOMO, 2: LUMO, 3: band gap, 4: Voc, 5: Jsc, 6: FF')
parser.add_argument('-splitter', type=str, default='scaffold')
parser.add_argument('-model', type=str, default='GINE')
parser.add_argument('-num_trial', type=int, default=5)
parser.add_argument('-gpu', type=int, default=1)

parser.add_argument('-num_epoch', type=int, default=100)
parser.add_argument('-eval_freq', type=int, default=10)
parser.add_argument('-batch_size', type=int, default=128)
parser.add_argument('-bn', type=bool, default=False)
parser.add_argument('-norm', type=str, default=None, choices=[None, 'BatchNorm', 'LayerNorm'])

parser.add_argument('-lr', type=float, default=0.001)
parser.add_argument('-weight_decay', type=float, default=5e-4)
parser.add_argument('-dropout', type=float, default=0.0)
parser.add_argument('-criterion', type=str, default='MAE')
parser.add_argument('-scheduler', type=str, default='onecycle-0.05')
parser.add_argument('-final_div_factor', type=float, default=1e4)


parser.add_argument('-num_layer', type=int, default=5)
parser.add_argument('-hidden_dim', type=int, default=128)
parser.add_argument('-l2norm', type=bool, default=False)
parser.add_argument('-pool', type=str, default='add')
parser.add_argument('-jk', type=str, default='cat')
parser.add_argument('-first_residual', type=bool, default=True)
parser.add_argument('-residual', type=bool, default=False)

parser.add_argument('-transform', type=str, default=None, choices=[None, 'VirtualNode'])
parser.add_argument('-best_val', type=bool, default=True)
parser.add_argument('-final_batch_norm', type=bool, default=False)

parser.set_defaults(model='AttentiveFP')
for target_task in [0]:
    parser.set_defaults(target_task=target_task)
    for dataset in ['NFAs_51K','NFAs_1.2K','HOPV', 'PolymerFA', 'nNFA', 'pNFA']:
        parser.set_defaults(dataset=dataset)
        if dataset == 'pNFA':
            parser.set_defaults(normalize=True)
        else:
            parser.set_defaults(normalize=False)    
        for dim in [512]:
            parser.set_defaults(hidden_dim=dim) 
            for num_layer in [8]:
                parser.set_defaults(num_layer=num_layer)
                for dropout in [0.5]:
                    parser.set_defaults(dropout=dropout)   
                    for bs in [32]:
                        parser.set_defaults(batch_size=bs)
                        for lr in [1e-3, 5e-4, 1e-4, 5e-5]:
                            parser.set_defaults(lr=lr)       
                            args=[]
                            args = parser.parse_args(args=args)                
                            with torch.cuda.device(f'cuda:{args.gpu}'):
                                torch.cuda.empty_cache()
                            train(args, None)